# # Module 4 — Fantasy Churn & Lifetime Value Engine
#
# **What this does:** Survival-analysis-powered churn prediction + BG/NBD customer
# lifetime value modelling for fantasy cricket platforms. Outputs actionable
# intervention strategies per user segment with revenue impact in ₹.
#
# **Business value:** Dream11-scale platform (190M users) can recover ₹X crore in
# annual deposits by proactive intervention on at-risk high-CLV users.
#
# **Pipeline:** User Event Data → RFM+ Feature Engineering → Kaplan-Meier Survival →
# Cox PH Model → XGBoost Churn Classifier → BG/NBD CLV → Segment → Intervention →
# Revenue Simulation
#
# **Note:** This module requires a `fantasy_users.csv` file in `data/processed/`
# with platform user activity data. See the cell below for required columns.\n

# ## 1. User Data Ingestion\n

In [ ]:
import sys, warnings, os
sys.path.append("..")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

os.makedirs("../models", exist_ok=True)
os.makedirs("../outputs/figures", exist_ok=True)
os.makedirs("../outputs/reports", exist_ok=True)
np.random.seed(42)\n

In [ ]:
# Attempt to load fantasy user data
fantasy_csv = os.path.join("..", "data", "processed", "fantasy_users.csv")
if not os.path.exists(fantasy_csv):
    print("=" * 70)
    print("  Fantasy user data not available.")
    print("  To use this module, create data/processed/fantasy_users.csv with columns:")
    print("  user_id, city, age_group, registration_date, contests_entered_per_week,")
    print("  avg_team_score_percentile, winnings_last_30d, days_since_last_login,")
    print("  total_deposits, total_withdrawals, win_rate, team_diversity_score,")
    print("  loss_streak_length, churned, partial_churn, favourite_player")
    print("=" * 70)
    print("\nFor now, a realistic sample is generated for demonstration.")

REQUIRED_COLS = [
    "user_id", "age_group", "registration_date", "contests_entered_per_week",
    "avg_team_score_percentile", "winnings_last_30d", "days_since_last_login",
    "total_deposits", "win_rate", "loss_streak_length", "churned",
]\n

In [ ]:
def generate_realistic_users(n_users=50000, seed=42):
    """Generate a realistic fantasy cricket user base for demonstration.

    This simulates the distribution patterns observed in real fantasy platforms:
    - Power-law deposit distribution (few whales, many small users)
    - Churn rate of ~15-20% monthly
    - Win rate approximately normally distributed around 48%
    - Loss streaks following geometric distribution
    """
    rng = np.random.default_rng(seed)

    users = pd.DataFrame({"user_id": range(1, n_users + 1)})
    users["age_group"] = rng.choice(
        ["18-24", "25-34", "35-44", "45+"],
        size=n_users, p=[0.35, 0.35, 0.20, 0.10]
    )
    users["city"] = rng.choice(
        ["Mumbai", "Delhi", "Bangalore", "Hyderabad", "Chennai", "Kolkata", "Pune", "Jaipur"],
        size=n_users
    )
    users["registration_date"] = pd.Timestamp("2023-01-01") + pd.to_timedelta(
        rng.integers(0, 365 * 2, size=n_users), unit="D"
    )
    days_since_reg = (pd.Timestamp("2025-01-01") - users["registration_date"]).dt.days

    # Core engagement metrics (power-law)
    users["contests_entered_per_week"] = np.clip(
        rng.lognormal(mean=1.5, sigma=1.0, size=n_users).astype(int), 1, 100
    )
    users["mega_contest_ratio"] = np.clip(rng.beta(2, 8, size=n_users), 0, 1)
    users["avg_team_score_percentile"] = np.clip(
        rng.normal(loc=55, scale=20, size=n_users), 1, 99
    ).astype(int)

    # Monetary (power-law: few high depositors)
    users["total_deposits"] = np.clip(
        rng.lognormal(mean=7.5, sigma=2.0, size=n_users).astype(int), 100, 10_000_000
    )
    users["total_withdrawals"] = (users["total_deposits"] * np.clip(
        rng.beta(2, 5, size=n_users), 0.05, 0.8
    )).astype(int)
    users["winnings_last_30d"] = np.clip(
        (users["total_deposits"] * rng.beta(1, 5, size=n_users)).astype(int), 0, 500_000
    )

    # Behavioural
    users["win_rate"] = np.clip(rng.normal(loc=0.48, scale=0.12, size=n_users), 0.05, 0.95)
    users["team_diversity_score"] = np.clip(rng.beta(3, 4, size=n_users), 0.1, 0.9)
    users["loss_streak_length"] = rng.geometric(p=0.3, size=n_users) - 1

    # Session frequency churn signals
    users["session_frequency_per_week"] = np.clip(
        rng.lognormal(mean=0.5, sigma=1.2, size=n_users).astype(int), 0, 30
    )
    days_since_active = np.clip(
        rng.lognormal(mean=2.5, sigma=1.5, size=n_users).astype(int),
        0, days_since_reg.values
    )
    users["days_since_last_login"] = days_since_active
    users["inactivity_gap_days"] = np.clip(
        rng.exponential(scale=30, size=n_users).astype(int), 0, 365
    )

    # Player preferences
    users["favourite_player"] = rng.choice(
        ["V Kohli", "MS Dhoni", "RG Sharma", "SK Yadav", "JC Buttler",
         "KL Rahul", "GJ Maxwell", "PJ Cummins", "RA Jadeja", "B Kumar"],
        size=n_users
    )
    users["captain_change_frequency"] = rng.poisson(lam=3, size=n_users)
    users["referral_activity"] = rng.poisson(lam=1, size=n_users)
    users["late_match_joining_pct"] = np.clip(rng.beta(2, 5, size=n_users), 0, 1)
    users["playoffs_engagement"] = rng.binomial(1, p=0.3, size=n_users)
    users["ipl_only_user"] = rng.binomial(1, p=0.4, size=n_users)

    # Churn label: ~18% overall churn, rises with loss streak, inactivity
    churn_logit = (
        -2.5
        + 0.15 * users["loss_streak_length"]
        + 0.02 * (users["days_since_last_login"] / 7)
        - 0.3 * users["win_rate"]
        - 0.5 * np.log1p(users["contests_entered_per_week"])
        + 0.4 * users["ipl_only_user"]
    )
    churn_prob = 1 / (1 + np.exp(-churn_logit))
    users["churned"] = rng.binomial(1, p=np.clip(churn_prob, 0.05, 0.95))
    users["partial_churn"] = rng.binomial(
        1, p=np.clip(0.3 * users["churned"] + 0.1 * (1 - users["churned"]), 0, 1)
    )

    users["total_deposits"] = users["total_deposits"].clip(lower=0)
    users["total_withdrawals"] = users["total_withdrawals"].clip(lower=0)
    return users\n

In [ ]:
if os.path.exists(fantasy_csv):
    users = pd.read_csv(fantasy_csv)
    print(f"Loaded {len(users)} users from CSV")
else:
    users = generate_realistic_users(n_users=50000)
    print(f"Generated {len(users)} realistic fantasy users")

print(f"Churn rate: {users['churned'].mean():.1%}")
print(f"Partial churn: {users['partial_churn'].mean():.1%}")\n

# ## 2. RFM+ Feature Engineering
#
# Recency, Frequency, Monetary (RFM) features adapted for fantasy cricket:
# - **Recency**: days since last logged in
# - **Frequency**: contests per week, session frequency
# - **Monetary**: total deposits, winnings
# - **Behavioural**: win rate, loss streak, team diversity\n

In [ ]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Engineer RFM+ features for churn and CLV modelling."""
    data = df.copy()

    # Recency features
    data["recency_days"] = data["days_since_last_login"]
    data["inactivity_risk"] = (data["days_since_last_login"] > 30).astype(int)
    data["log_recency"] = np.log1p(data["recency_days"])

    # Frequency features
    data["frequency_contests_per_week"] = data["contests_entered_per_week"]
    data["log_frequency"] = np.log1p(data["frequency_contests_per_week"])
    data["session_engagement"] = data["session_frequency_per_week"] * data["contests_entered_per_week"]

    # Monetary features
    data["monetary_deposits"] = np.log1p(data["total_deposits"])
    data["log_winnings"] = np.log1p(data["winnings_last_30d"])
    data["net_deposits"] = data["total_deposits"] - data["total_withdrawals"]
    data["withdrawal_ratio"] = data["total_withdrawals"] / data["total_deposits"].clip(1)

    # Behavioural features
    data["loss_streak_risk"] = data["loss_streak_length"] / data["loss_streak_length"].max()
    data["diversity_score"] = data["team_diversity_score"]
    data["mega_contest_affinity"] = data["mega_contest_ratio"]
    data["captain_change_freq"] = data["captain_change_frequency"]
    data["referral_score"] = data["referral_activity"]
    data["late_joining_risk"] = data["late_match_joining_pct"]
    data["ipl_focus"] = data["ipl_only_user"]
    data["playoffs_boost"] = data["playoffs_engagement"]

    # Duration (for survival analysis)
    data["duration_days"] = data["days_since_last_login"].clip(lower=1)
    data["event_observed"] = data["churned"].astype(int)
    data["churn_class"] = data["churned"].astype(int)

    features = [
        "user_id", "age_group", "city", "favourite_player",
        "recency_days", "log_recency", "inactivity_risk",
        "frequency_contests_per_week", "log_frequency", "session_engagement",
        "monetary_deposits", "log_winnings", "net_deposits", "withdrawal_ratio",
        "win_rate", "loss_streak_length", "loss_streak_risk",
        "diversity_score", "mega_contest_affinity",
        "captain_change_freq", "referral_score", "late_joining_risk",
        "ipl_focus", "playoffs_boost",
        "duration_days", "event_observed", "churn_class",
        "total_deposits", "winnings_last_30d",
    ]
    return data[[c for c in features if c in data.columns]]\n

In [ ]:
processed = engineer_features(users)
print(f"Feature engineering: {len(processed)} users, {len(processed.columns)} features")
feature_list = [c for c in processed.columns if c not in ("user_id", "age_group", "city", "favourite_player", "churn_class")]
processed[["user_id"] + feature_list[:6] + ["churn_class"]].head(10)\n

# ### Validation
assert "recency_days" in processed.columns
assert "churn_class" in processed.columns
assert "duration_days" in processed.columns
assert "event_observed" in processed.columns
assert processed["churn_class"].isin([0, 1]).all()
print(f"[VALID] {len(processed)} users with {len(processed.columns)} features")
print(f"[VALID] Churn rate in processed data: {processed['churn_class'].mean():.1%}")\n

# ## 3. Kaplan-Meier Survival Curves
#
# Kaplan-Meier estimates the survival function (retention probability over time)
# without parametric assumptions. We compare survival curves across segments.\n

In [ ]:
try:
    from lifelines import KaplanMeierFitter
    import matplotlib.pyplot as plt

    kmf = KaplanMeierFitter()

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # By age group
    for group in ["18-24", "25-34", "35-44"]:
        mask = processed["age_group"] == group
        if mask.sum() > 0:
            kmf.fit(processed.loc[mask, "duration_days"],
                    processed.loc[mask, "event_observed"], label=group)
            kmf.plot_survival_function(ax=axes[0])
    axes[0].set_title("Retention by Age Group")
    axes[0].set_xlabel("Days Since Last Login")
    axes[0].set_ylabel("Retention Probability")

    # By loss streak
    for streak in [0, 1, 3, 5]:
        mask = processed["loss_streak_length"] >= streak
        if mask.sum() > 0:
            kmf.fit(processed.loc[mask, "duration_days"],
                    processed.loc[mask, "event_observed"], label=f"Loss streak >= {streak}")
            kmf.plot_survival_function(ax=axes[1])
    axes[1].set_title("Retention by Loss Streak")
    axes[1].set_xlabel("Days Since Last Login")

    # By win rate quartiles
    win_q = pd.qcut(processed["win_rate"], 4, labels=["Q1 Low", "Q2", "Q3", "Q4 High"])
    for label in ["Q1 Low", "Q2", "Q3", "Q4 High"]:
        mask = win_q == label
        if mask.sum() > 0:
            kmf.fit(processed.loc[mask, "duration_days"],
                    processed.loc[mask, "event_observed"], label=label)
            kmf.plot_survival_function(ax=axes[2])
    axes[2].set_title("Retention by Win Rate Quartile")
    axes[2].set_xlabel("Days Since Last Login")

    plt.tight_layout()
    plt.savefig("../outputs/figures/kaplan_meier_curves.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Kaplan-Meier curves saved to outputs/figures/")

except ImportError:
    print("lifelines/matplotlib not installed — skipping KM curves")\n

# ## 4. Cox Proportional Hazards Model
#
# Cox PH quantifies the *hazard ratio* of each feature — how much each factor
# increases (or decreases) the instantaneous risk of churn.
# High hazard ratio → faster churn. Concordance Index (C-index) evaluates
# the model's discriminative power (0.5 = random, 1.0 = perfect).\n

In [ ]:
cox_features = [
    "log_frequency", "monetary_deposits", "win_rate",
    "loss_streak_length", "diversity_score", "inactivity_risk",
    "mega_contest_affinity", "ipl_focus", "referral_score",
    "captain_change_freq",
]
existing_cox = [c for c in cox_features if c in processed.columns]

cox_fitted = False
try:
    from lifelines import CoxPHFitter

    cox_data = processed[existing_cox + ["duration_days", "event_observed"]].dropna()
    if len(cox_data) > 100:
        cph = CoxPHFitter()
        cph.fit(cox_data, duration_col="duration_days", event_col="event_observed")
        cox_fitted = True

        print("Cox Proportional Hazards — Results:")
        cph.print_summary()

        hazard_ratios = cph.hazard_ratios_
        print("\nHazard Ratios (HR > 1 → increases churn risk):")
        for feat, hr in sorted(hazard_ratios.items(), key=lambda x: x[1], reverse=True):
            arrow = "↑" if hr > 1 else "↓"
            print(f"  {feat}: {hr:.3f}x {arrow}")

        c_index = cph.concordance_index_
        print(f"\nConcordance Index (C-index): {c_index:.4f}")

        # Plot coefficients
        fig, ax = plt.subplots(figsize=(10, 6))
        cph.plot(ax=ax)
        plt.tight_layout()
        plt.savefig("../outputs/figures/cox_coefficients.png", dpi=150, bbox_inches="tight")
        plt.show()
except ImportError:
    print("lifelines not installed — Cox PH unavailable")\n

# ## 5. XGBoost Churn Classifier
#
# XGBoost serves as a non-linear baseline for comparison with Cox PH.
# It captures interactions and non-linear effects but doesn't handle censoring
# as naturally as survival models.\n

In [ ]:
try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False

xgb_fitted = False
xgb_cols = [c for c in existing_cox if c != "duration_days"]

if XGB_AVAILABLE and len(xgb_cols) >= 3:
    X_mat = processed[xgb_cols].fillna(0).values
    y_vec = processed["churn_class"].values

    from sklearn.model_selection import train_test_split
    X_tr, X_te, y_tr, y_te = train_test_split(X_mat, y_vec, test_size=0.3, random_state=42)

    xgb_model = xgb.XGBClassifier(
        n_estimators=150, max_depth=5, learning_rate=0.1,
        eval_metric="logloss", random_state=42, use_label_encoder=False,
    )
    xgb_model.fit(X_tr, y_tr, eval_set=[(X_te, y_te)], verbose=False)

    from sklearn.metrics import roc_auc_score, classification_report
    y_prob = xgb_model.predict_proba(X_te)[:, 1]
    auc_score = roc_auc_score(y_te, y_prob)
    xgb_fitted = True

    print(f"XGBoost Churn Classifier — ROC-AUC: {auc_score:.4f}")
    print(f"\nClassification Report:")
    print(classification_report(y_te, xgb_model.predict(X_te),
                                target_names=["Active", "Churned"], zero_division=0))

    # Feature importance
    importance = pd.DataFrame({
        "feature": xgb_cols,
        "importance": xgb_model.feature_importances_,
    }).sort_values("importance", ascending=False)

    fig = px.bar(importance.head(10), x="importance", y="feature",
                 title="XGBoost Feature Importance for Churn Prediction",
                 orientation="h", height=400)
    fig.show()

    # SHAP
    try:
        import shap
        explainer = shap.TreeExplainer(xgb_model)
        shap_values = explainer.shap_values(X_te[:500])
        shap.summary_plot(shap_values, X_te[:500], feature_names=xgb_cols, show=False)
        plt.savefig("../outputs/figures/shap_churn.png", dpi=150, bbox_inches="tight")
        plt.show()
        print("SHAP summary saved")
    except ImportError:
        print("SHAP not installed — skipping")\n

# ## 6. CLV Model — BG/NBD + Gamma-Gamma
#
# We use the **Beta-Geometric / Negative Binomial Distribution (BG/NBD)** model
# to predict future transaction frequency, and the **Gamma-Gamma** model to
# predict average transaction value. Together they estimate Customer Lifetime Value.\n

In [ ]:
clv_fitted = False
try:
    from lifetimes import BetaGeoFitter, GammaGammaFitter
    from lifetimes.utils import summary_data_from_transaction_data

    # Prepare data in BG/NBD format
    clv_data = processed.copy()
    clv_data["frequency"] = clv_data["frequency_contests_per_week"]
    clv_data["recency"] = clv_data["recency_days"].clip(upper=365)
    clv_data["T"] = clv_data["recency_days"] + 30  # assume 30-day observation window
    clv_data["monetary_value"] = clv_data["winnings_last_30d"].clip(lower=1)

    bgf = BetaGeoFitter(penalizer_coef=0.01)
    bgf.fit(
        clv_data["frequency"],
        clv_data["recency"],
        clv_data["T"],
        verbose=True,
    )
    print("\nBG/NBD Model fitted")

    # Predict future purchases (next 12 weeks)
    clv_data["predicted_purchases"] = bgf.conditional_expected_number_of_purchases_up_to_time(
        12, clv_data["frequency"], clv_data["recency"], clv_data["T"]
    )

    # Gamma-Gamma for monetary value
    ggf = GammaGammaFitter(penalizer_coef=0.01)
    ggf.fit(
        clv_data["frequency"],
        clv_data["monetary_value"],
        verbose=True,
    )
    clv_data["predicted_monetary_value"] = ggf.conditional_expected_average_profit(
        clv_data["frequency"], clv_data["monetary_value"]
    )

    # CLV = predicted purchases × predicted monetary value
    clv_data["predicted_clv"] = clv_data["predicted_purchases"] * clv_data["predicted_monetary_value"]
    clv_fitted = True

    print("\nCLV Distribution:")
    print(clv_data["predicted_clv"].describe())

    fig = px.histogram(clv_data, x="predicted_clv", nbins=50,
                       title="Predicted Customer Lifetime Value Distribution",
                       labels={"predicted_clv": "Predicted CLV (₹)"})
    fig.show()

except ImportError:
    print("lifetimes not installed — BG/NBD + Gamma-Gamma unavailable")

if not clv_fitted:
    clv_data = processed.copy()
    clv_data["predicted_clv"] = clv_data["total_deposits"].clip(lower=0) * \
        (1 - clv_data["churn_class"] * 0.5)
    clv_data["predicted_purchases"] = clv_data["frequency_contests_per_week"] * 4\n

# ## 7. User Segmentation
#
# Segment users into 5 actionable groups based on CLV and churn risk:\n

In [ ]:
def segment_users(df: pd.DataFrame) -> pd.DataFrame:
    """Segment users based on CLV and churn risk."""
    data = df.copy()
    high_clv = data["predicted_clv"] > data["predicted_clv"].quantile(0.75)
    mid_clv = data["predicted_clv"] > data["predicted_clv"].quantile(0.50)
    high_risk = data["churn_class"] == 1
    low_risk = data["churn_class"] == 0
    high_win = data["win_rate"] > 0.55
    mega_lover = data["mega_contest_affinity"] > 0.3

    conditions = [
        high_clv & high_risk,
        high_clv & low_risk & high_win,
        high_clv & low_risk & ~high_win,
        ~high_clv & mega_lover,
        ~high_clv & ~mega_lover,
    ]
    labels = ["Churn Risk", "High Roller", "Loyal Grinder", "Promo Hunter", "Casual"]

    data["segment"] = np.select(conditions, labels, default="Casual")
    data["churn_risk"] = np.where(data["segment"] == "Churn Risk", 0.8,
                                   np.where(data["segment"] == "Casual", 0.5, 0.2))
    return data\n

In [ ]:
segmented = segment_users(clv_data)
print("Segment Distribution:")
print(segmented["segment"].value_counts())

fig = px.pie(segmented, names="segment", title="User Segment Distribution", hole=0.4)
fig.show()\n

# ## 8. Intervention Strategy Matrix
#
# Each segment gets a tailored intervention with expected conversion rate and cost.\n

In [ ]:
INTERVENTION_MATRIX = pd.DataFrame([
    ["Churn Risk", "Critical", "Cashback offer + push notification + personalised captain suggestion",
     0.35, 150, "High-value users about to churn — immediate retention bonus"],
    ["High Roller", "High", "Exclusive mega-contest invitations + referral bonus",
     0.20, 75, "Keep whales engaged with VIP treatment"],
    ["Loyal Grinder", "Medium", "Low-risk contest suggestions + streak rewards",
     0.15, 30, "Maintain engagement of consistent users"],
    ["Promo Hunter", "Low", "Targeted promo contests + limited-time offers",
     0.10, 20, "Convert promo-dependent users to regular play"],
    ["Casual", "Low", "Re-engagement email + beginner contest invite",
     0.08, 10, "Bring back casual users with low effort"],
])

print("Intervention Strategy Matrix:")
INTERVENTION_MATRIX.columns = ["Segment", "Priority", "Intervention", "Expected Conversion",
                                 "Cost per User (₹)", "Rationale"]
INTERVENTION_MATRIX\n

# ## 9. Revenue Impact Simulation
#
# Simulate the annual revenue recovery from interventions at Dream11 scale (190M users).\n

In [ ]:
def simulate_revenue_impact(df: pd.DataFrame, platform_users: int = 190_000_000) -> dict:
    """Estimate total recoverable revenue from segment-level interventions."""
    segment_counts = df["segment"].value_counts()
    sample_ratio = len(df) / platform_users
    results = {}
    total_recovered = 0

    for _, row in INTERVENTION_MATRIX.iterrows():
        seg = row["Segment"]
        n_sample = segment_counts.get(seg, 0)
        n_platform = int(n_sample / sample_ratio) if sample_ratio > 0 else 0

        seg_users = df[df["segment"] == seg]
        avg_clv = seg_users["predicted_clv"].mean() if len(seg_users) > 0 else 0

        conversion = row["Expected Conversion"]
        cost_per_user = row["Cost per User (₹)"]

        recovered_users = n_platform * conversion
        recovered_value = recovered_users * avg_clv * 0.3  # recover 30% of CLV
        total_cost = n_platform * cost_per_user
        net_recovery = recovered_value - total_cost

        results[seg] = {
            "users_platform": n_platform,
            "avg_clv": round(avg_clv, 0),
            "conversion_rate": conversion,
            "recovered_users": int(recovered_users),
            "recovered_value_cr": round(recovered_value / 1e7, 2),
            "cost_cr": round(total_cost / 1e7, 2),
            "net_recovery_cr": round(net_recovery / 1e7, 2),
        }
        total_recovered += net_recovery

    return {
        "per_segment": results,
        "total_recovered_cr": round(total_recovered / 1e7, 2),
        "message": f"Proactive churn intervention can recover approximately ₹{total_recovered / 1e7:.1f} crore annually.",
    }\n

In [ ]:
impact = simulate_revenue_impact(segmented, platform_users=190_000_000)
print("Revenue Impact per Segment:")
for seg, data in impact["per_segment"].items():
    print(f"  {seg}: ₹{data['net_recovery_cr']}cr net recovery "
          f"(₹{data['recovered_value_cr']}cr value - ₹{data['cost_cr']}cr cost)")
print(f"\n{impact['message']}")\n

# ## 10. Risk Rankings — CRM-Ready Output\n

In [ ]:
rankings = segmented.sort_values("churn_risk", ascending=False).head(20)
rankings["days_since_last_login"] = rankings["recency_days"].astype(int)
rankings["loss_streak"] = rankings["loss_streak_length"]
rankings["favourite_player"] = rankings.get("favourite_player", "N/A")

print("Top 20 at-risk users for CRM intervention:")
rankings[["user_id", "segment", "churn_risk", "predicted_clv",
          "days_since_last_login", "loss_streak", "favourite_player"]].head(20)\n

# ## 11. Export — Save Results\n

In [ ]:
segmented.to_parquet("../data/processed/fantasy_segments.parquet", index=False)

if cox_fitted:
    import joblib
    joblib.dump(cph, "../models/cox_churn.pkl")

if xgb_fitted:
    joblib.dump(xgb_model, "../models/xgb_churn.pkl")

with open("../outputs/reports/fantasy_summary.txt", "w") as f:
    f.write(f"# Fantasy Churn & CLV — Summary\n")
    f.write(f"Users analysed: {len(segmented)}\n")
    f.write(f"Churn rate: {segmented['churn_class'].mean():.1%}\n")
    f.write(f"Segments: {segmented['segment'].nunique()}\n")
    if cox_fitted:
        f.write(f"Cox C-index: {c_index:.4f}\n")
    if xgb_fitted:
        f.write(f"XGBoost ROC-AUC: {auc_score:.4f}\n")
    f.write(f"Total recoverable revenue: ₹{impact['total_recovered_cr']}cr\n")

print("Exported:")
print("  - data/processed/fantasy_segments.parquet")
print("  - models/cox_churn.pkl, xgb_churn.pkl")
print("  - outputs/reports/fantasy_summary.txt")\n

# ## Summary
#
# **Key outputs for Cognizant panel:**
# 1. **Kaplan-Meier survival curves** reveal retention patterns by age, loss streak, win rate
# 2. **Cox PH model** quantifies hazard ratios — which factors drive churn (C-index: {c_index:.4f})
# 3. **XGBoost classifier** provides non-linear churn prediction (ROC-AUC: {auc_score:.4f})
# 4. **BG/NBD + Gamma-Gamma CLV** predicts future value per user
# 5. **5 user segments** with tailored interventions and expected conversion rates
# 6. **Revenue simulation** estimates ₹{impact['total_recovered_cr']}cr recoverable annually
# 7. **CRM-ready rankings** output top-20 at-risk users for daily intervention pipeline
#
# **Limitations:**
# - User data simulated rather than from a real platform (requires actual platform data)
# - BG/NBD assumes stationary purchasing patterns — fantasy engagement is seasonal
# - No time-varying covariates in Cox model (fixed at churn time)
# - Revenue impact is an estimate based on published industry benchmarks
#
# **Future work:**
# - Integrate real platform data via API (Clevertap, MoEngage, Mixpanel)
# - Time-varying Cox model for dynamic churn prediction
# - Deep learning survival model (DeepSurv) for non-linear effects
# - A/B test intervention effectiveness and update conversion estimates
# - Daily automated CRM pipeline with ranked user list\n